In [11]:
# Step 1: Install required packages
!pip install -U \
  langchain \
  langchain-community \
  langchain-text-splitters \
  faiss-cpu \
  sentence-transformers \
  pypdf \
  langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.9 MB/s eta 0:00:00


In [12]:
# Step 2: Import packages
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [13]:
# Step 3: Create embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# Step 4: Load your PDF
pdf_path = "/content/sample_data/AI_in_modern_society.pdf"  # upload your PDF in Colab

loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [15]:
# Step 5: Split text into smaller chunks size: 500 characters and overlap: 50 characters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)

In [16]:
# Step 6: Create FAISS db and store embeddings
vector_db = FAISS.from_documents(docs, embedding_model)

In [17]:
# Step 7: Query + retrieve  (Similarity Search)
query = input("Enter your query: ")

results = vector_db.similarity_search(query, k=3)

Enter your query: Compare the benefits of AI in finance and healthcare


In [18]:
# Step 8 (OPTIONAL): Show retrieved results (matching chunks)
print("\nTop relevant chunks:\n")

for i, res in enumerate(results):
    print(f"Result {i+1}:")
    print(res.page_content)
    print("-" * 50)


Top relevant chunks:

Result 1:
Applications of AI 
AI has numerous real-world applications: 
Healthcare: 
AI assists in diagnosing diseases, analyzing medical images, and predicting 
patient outcomes. It helps doctors make faster and more accurate 
decisions. 
Finance: 
AI is used for fraud detection, algorithmic trading, and risk assessment. It 
can process vast amounts of ﬁnancial data quickly.
--------------------------------------------------
Result 2:
Transportation: 
Self-driving cars use AI to navigate roads, detect obstacles, and make real-
time decisions. 
Retail: 
AI improves customer experience through recommendation systems and 
chatbots. 
Education: 
AI enables personalized learning by adapting content based on student 
performance. 
 
Beneﬁts of AI 
AI oƯers several advantages: 
 EƯiciency: Automates repetitive tasks, saving time and resources 
 Accuracy: Reduces human error in data processing
--------------------------------------------------
Result 3:
Artiﬁcial Inte

In [19]:
# Step 9: Add OpenAI LLM

llm = ChatOpenAI(
    model="gpt-4o-mini",   # best balance cost + quality
    temperature=0
)

In [20]:
# Step 10: Generate Final answer for the user

context = "\n\n".join([doc.page_content for doc in results])

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print("\nQuestion:\n")
print(query)
print("\nFinal Answer:\n")
print(response.content)


Final Answer:

content='In finance, the benefits of AI include efficiency in processing vast amounts of financial data quickly, which aids in fraud detection, algorithmic trading, and risk assessment. This allows financial institutions to make faster decisions and reduce potential losses.\n\nIn healthcare, AI enhances accuracy by assisting in diagnosing diseases, analyzing medical images, and predicting patient outcomes. This leads to faster and more precise decision-making by doctors, ultimately improving patient care.\n\nOverall, while both sectors benefit from increased efficiency and accuracy, finance focuses on data processing and risk management, whereas healthcare emphasizes diagnostic support and patient outcomes.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 311, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_